In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import cross_validate, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier
import joblib

**Explanation :** This cell loads the cleaned dataset from a CSV file and removes missing values to ensure that the data used for analysis and modeling is complete.

In [2]:
cleaned_data = pd.read_csv("cleaned_dataset_final.csv").dropna()

## Encode evey STR

**Explanation :** This cell selects the relevant categorical columns from the dataset (such as product name, aisle, and department) that will be used for further preprocessing or feature engineering.

In [3]:
cols = ["product_name", "aisle", "department"]

for col in cols:
    codes = pd.factorize(cleaned_data[col])[0]
    cleaned_data[col] = (codes / codes.max()) * 20 + 1
cleaned_data.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,product_name,aisle,department,add_to_cart_order,reordered
0,2,202279.0,5.0,9.0,8.0,0.0,33120.0,1.00000,1.000000,1.000000,1.0,1.0
1,2,202279.0,5.0,9.0,8.0,0.0,28985.0,1.00044,1.149254,1.952381,2.0,1.0
2,2,202279.0,5.0,9.0,8.0,0.0,9327.0,1.00088,1.298507,2.904762,3.0,0.0
3,2,202279.0,5.0,9.0,8.0,0.0,45918.0,1.00132,1.447761,2.904762,4.0,1.0
4,2,202279.0,5.0,9.0,8.0,0.0,30035.0,1.00176,1.597015,2.904762,5.0,0.0


## Reduce high numbers

**Explanation :** This cell selects user and product identifiers from the dataset, which may be used to build interactions between users and products for recommendation or analysis.

In [4]:
cols = ["user_id", "product_id"]

for col in cols:
  codes = pd.factorize(cleaned_data[col])[0]
  cleaned_data[col] = (codes / codes.max()) * 50 + 1
cleaned_data.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,product_name,aisle,department,add_to_cart_order,reordered
0,2,1.0,5.0,9.0,8.0,0.0,1.000000,1.00000,1.000000,1.000000,1.0,1.0
1,2,1.0,5.0,9.0,8.0,0.0,1.001100,1.00044,1.149254,1.952381,2.0,1.0
2,2,1.0,5.0,9.0,8.0,0.0,1.002200,1.00088,1.298507,2.904762,3.0,0.0
3,2,1.0,5.0,9.0,8.0,0.0,1.003301,1.00132,1.447761,2.904762,4.0,1.0
4,2,1.0,5.0,9.0,8.0,0.0,1.004401,1.00176,1.597015,2.904762,5.0,0.0


## LogisticRegression Model

**Explanation :** This cell defines the feature variables (X) and the target variable (y) that will be used to train the machine learning model.

The logistic regression model itself simply models probability of output in terms of input and does not perform statistical classification (it is not a classifier), though it can be used to make a classifier, for instance by choosing a cutoff value and classifying inputs with probability greater than the cutoff as one class, below the cutoff as the other; this is a common way to make a binary classifier.

In [5]:
# Sélection des features et de la target
features = [
    "user_id", "order_number", "order_dow", "order_hour_of_day",
    "days_since_prior_order", "product_id", "product_name", "aisle", "department",
    "add_to_cart_order"
]
x = cleaned_data[features]
y = cleaned_data["reordered"]

# Optionnel : échantillon plus petit pour tester plus rapidement
x_small = x.sample(50000, random_state=42) 
y_small = y.loc[x_small.index]

# Initialisation du modèle
model = LogisticRegression(max_iter=1000)

# Cross-validation sur tout le dataset
scores = cross_val_score(model, x_small, y_small, cv=5)
print("Cross-validation scores :", scores)
print("Mean score :", scores.mean())


Cross-validation scores : [0.6219 0.6303 0.6237 0.6324 0.6315]
Mean score : 0.6279600000000001


## LogisticRegression Building Model

**Explanation :** This cell splits the dataset into training and testing sets using train_test_split. The training data is used to train the model, while the test data is used to evaluate its performance.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(x_small, y_small, test_size=0.2, random_state=42)

y_test_backup = y_test

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

logr = LogisticRegression()
logr.fit(X_train_scaled, y_train)
joblib.dump(logr, "model_logistic_regression.pkl")

['model_logistic_regression.pkl']

## LogisticRegression Model Evaluation 

**Explanation :** This cell defines a function used to evaluate machine learning models. The function calculates performance metrics based on the predictions made by the model on the test dataset.

In [7]:
def model_evaluation(model, X_test, y_test, model_name):
  y_pred = model.predict(X_test)
  MAE = mean_absolute_error(y_test, y_pred)
  MSE = mean_squared_error(y_test, y_pred)
  RMSE = np.sqrt(MSE)
  R2_Score = r2_score(y_test, y_pred)
  return pd.DataFrame(
      [MAE, MSE, RMSE, R2_Score],
      index=["MAE", "MSE", "RMSE", "R2-Score"],
      columns=[model_name],
  )

model_evaluation(logr, X_test_scaled, y_test, "Logistic Regression")

,Logistic Regression
MAE,0.373700
MSE,0.373700
RMSE,0.611310
R2-Score,-0.536323


## GradientBoostingClassifier Model Build

**Explanation :** This cell splits the dataset into training and testing sets using train_test_split. The training data is used to train the model, while the test data is used to evaluate its performance.

Gradient boosting is a machine learning technique based on boosting in a functional space, where the target is pseudo-residuals instead of residuals as in traditional boosting. It gives a prediction model in the form of an ensemble of weak prediction models, i.e., models that make very few assumptions about the data, which are typically simple decision trees.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    x_small, y_small, test_size=0.2, random_state=42
)

y_test_backup = y_test

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
joblib.dump(gb, "model_gradient_boosting.pkl")

['model_gradient_boosting.pkl']

## GradientBoostingClassifier Model Evaluation

**Explanation :** This cell defines a function used to evaluate machine learning models. The function calculates performance metrics based on the predictions made by the model on the test dataset.

In [9]:
def model_evaluation(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    MAE = mean_absolute_error(y_test, y_pred)
    MSE = mean_squared_error(y_test, y_pred)
    RMSE = np.sqrt(MSE)
    R2_Score = r2_score(y_test, y_pred)
    return pd.DataFrame(
        [MAE, MSE, RMSE, R2_Score],
        index=["MAE", "MSE", "RMSE", "R2-Score"],
        columns=[model_name],
    )

model_evaluation(gb, X_test, y_test, "Gradient Boosting")

,Gradient Boosting
MAE,0.315500
MSE,0.315500
RMSE,0.561694
R2-Score,-0.297056


## Preparation values for each model

**Explanation :** This cell groups the dataset by order ID to create a basket-level dataset, aggregating product information for each order. This step is useful for analyzing customer purchasing behavior.

In [10]:
basket_df = cleaned_data.groupby("order_id").agg({
  "order_dow": "first",
  "order_hour_of_day": "first",
  "days_since_prior_order": "first",
  "product_id": "count"
}).rename(columns={"product_id": "basket_size"})

X = basket_df[[
  "order_dow",
  "order_hour_of_day",
  "days_since_prior_order"
]]

y = basket_df["basket_size"]

print(X.shape, y.shape)

(445263, 3) (445263,)


## LinearRegression Model

**Explanation :** This cell initializes a Linear Regression model that will be trained to predict the target variable based on the selected features.

LinearRegression fits a linear model with coefficients w = (w1, …, wp) to minimize the residual sum of squares between the observed targets in the dataset, and the targets predicted by the linear approximation.

The MAE measures the average error in absolute value, without taking into account the direction of the error (positive or negative). Smaller the MAE value is, the better it is to see the margin of error and the best accuracy. So it's important to chose a model with the smalest MAE value.

The MSE value is the arithmetic mean of the squared differences between the model predictions and observed values. It is the value to be minimized within the field of a simple or multiple regression (see least squares).

The RSME measures the average difference between a statistical model's predicted values and the actual values.

the R2-Score is a measure of the quality of the prediction of a linear regression. The closer it is to 1, the better the prediction will be.

It's important to compare the MAE and the R2-Score to chose the model we are going to use for our project.

The MAE score is approximatly 9.03 and the R2-Score is approximatly 0.21 which is not bad.

In [11]:
linear_reg_cv = LinearRegression()
scaler = StandardScaler()
pipeline = make_pipeline(StandardScaler(), LinearRegression())

kf = KFold(n_splits=6, shuffle=True, random_state=0)
scoring = [
    "neg_mean_absolute_error",
    "neg_mean_squared_error",
    "neg_root_mean_squared_error",
    "r2",
]
result = cross_validate(pipeline, X, y, cv=kf, return_train_score=True, scoring=scoring)

MAE_mean = (-result["test_neg_mean_absolute_error"]).mean()
MAE_std = (-result["test_neg_mean_absolute_error"]).std()
MSE_mean = (-result["test_neg_mean_squared_error"]).mean()
MSE_std = (-result["test_neg_mean_squared_error"]).std()
RMSE_mean = (-result["test_neg_root_mean_squared_error"]).mean()
RMSE_std = (-result["test_neg_root_mean_squared_error"]).std()
R2_Score_mean = result["test_r2"].mean()
R2_Score_std = result["test_r2"].std()

pd.DataFrame(
    {
        "Mean": [MAE_mean, MSE_mean, RMSE_mean, R2_Score_mean],
        "Std": [MAE_std, MSE_std, RMSE_std, R2_Score_std],
    },
    index=["MAE", "MSE", "RMSE", "R2-Score"],
)

,Mean,Std
MAE,5.678391,0.017780
MSE,56.200319,0.379745
RMSE,7.496644,0.025368
R2-Score,0.002933,0.000411


## RandomForestRegressor Model

**Explanation :** This cell initializes a Random Forest Regressor model, which is an ensemble machine learning algorithm that uses multiple decision trees to improve prediction accuracy.

A random forest is a meta estimator that fits a number of decision tree regressors on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting.

In [12]:
model = RandomForestRegressor(n_jobs=-1)
kf = KFold(n_splits=6, shuffle=True, random_state=0)
scoring = [
    "neg_mean_absolute_error",
    "neg_mean_squared_error",
    "neg_root_mean_squared_error",
    "r2",
]
result = cross_validate(model, X, y, cv=kf, scoring=scoring, n_jobs=-1)

metrics = {
    "MAE": -result["test_neg_mean_absolute_error"],
    "MSE": -result["test_neg_mean_squared_error"],
    "RMSE": -result["test_neg_root_mean_squared_error"],
    "R2-Score": result["test_r2"],
}

df = pd.DataFrame(
    {m: [v.mean(), v.std()] for m, v in metrics.items()},
    index=["Mean", "Std"]
).T
df

,Mean,Std
MAE,5.599128,0.017707
MSE,55.044396,0.377758
RMSE,7.419147,0.025483
R2-Score,0.023441,0.000722


## Building LinearRegression model

**Explanation :** This cell performs another train-test split on the dataset to prepare the features and target variables for training and evaluating the machine learning model.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

y_test_backup = y_test

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
joblib.dump(lr, "model_linear_regression.pkl")

['model_linear_regression.pkl']

## LinearRegression Evaluation

**Explanation :** This cell defines a model evaluation function that takes a trained model and test data as input, then computes evaluation metrics to assess the model’s predictive performance.

In [14]:
def model_evaluation(model, X_test, y_test, model_name):
  y_pred = model.predict(X_test)
  MAE = mean_absolute_error(y_test, y_pred)
  MSE = mean_squared_error(y_test, y_pred)
  RMSE = np.sqrt(MSE)
  R2_Score = r2_score(y_test, y_pred)
  return pd.DataFrame(
    [MAE, MSE, RMSE, R2_Score],
    index=["MAE", "MSE", "RMSE", "R2-Score"],
    columns=[model_name],
  )

model_evaluation(lr, X_test_scaled, y_test, "Linear Regression")

,Linear Regression
MAE,5.671793
MSE,56.146621
RMSE,7.493105
R2-Score,0.002792


## Building RandomForestRegressor model

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

y_test_backup = y_test

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

rfr = RandomForestRegressor()
rfr.fit(X_train_scaled, y_train)
joblib.dump(rfr, "model_random_forest.pkl")

['model_random_forest.pkl']

## RandomForestRegressor Evaluation

In [16]:
def model_evaluation(model, X_test, y_test, model_name):
  y_pred = model.predict(X_test)
  MAE = mean_absolute_error(y_test, y_pred)
  MSE = mean_squared_error(y_test, y_pred)
  RMSE = np.sqrt(MSE)
  R2_Score = r2_score(y_test, y_pred)
  return pd.DataFrame(
    [MAE, MSE, RMSE, R2_Score],
    index=["MAE", "MSE", "RMSE", "R2-Score"],
    columns=[model_name],
  )

model_evaluation(rfr, X_test_scaled, y_test, "Random Forest Regressor")

,Random Forest Regressor
MAE,5.597509
MSE,55.025472
RMSE,7.417916
R2-Score,0.022704
